# Phase 9 — Nonequilibrium Thermodynamic Interpretation and Entropy-Production-Related Analysis

## Synthetic equilibrium vs nonequilibrium benchmark

Phase 9 begins with a stochastic system whose thermodynamic behavior is analytically known before any estimator is applied to the experimental MSC01 trajectories.

The benchmark is the two-dimensional rotational Ornstein–Uhlenbeck process

$$
dX = A X\,dt + \sqrt{2D}\,dW,
$$

with

$$
A = -kI + \omega R.
$$

The fixed baseline parameters are:

- \(k = 1\)
- \(D = 1\)
- \(dt = 0.1\)
- \(n_{\mathrm{steps}} = 100000\)
- seed \(= 2031\)

Two conditions are compared:

- equilibrium: \(\omega = 0\)
- nonequilibrium: \(\omega = 1\)

For both conditions, the stationary covariance is

$$
C = \frac{D}{k}I = I.
$$

Therefore, the two systems can have the same stationary spatial distribution even though their dynamics differ.

For the rotational nonequilibrium process, the analytical physical entropy-production rate is

$$
\sigma = \frac{2\omega^2}{k}.
$$

Thus the pre-specified analytical values are:

- equilibrium: \(\sigma = 0\)
- nonequilibrium: \(\sigma = 2\)

Before estimating entropy production from paths, this section tests two simpler properties:

1. whether both simulations reproduce the expected stationary covariance;
2. whether the nonequilibrium process displays the expected signed rotational probability current.

The rotational-current observable is

$$
c_t =
x_t y_{t+1}
-
y_t x_{t+1}.
$$

Its stationary expectation is

$$
E[c_t]
=
\frac{2D}{k}
e^{-k\,dt}
\sin(\omega\,dt).
$$

This rotational-current observable is **not** an entropy-production estimator.

In [1]:
import numpy as np
import pandas as pd

from _path import PROJECT_ROOT

from src.thermo import (
    analytic_epr_rotational_ou,
    analytic_mean_rotational_increment,
    rotational_increments,
    simulate_rotational_ou,
    stationary_covariance_isotropic,
)

print("Project root:", PROJECT_ROOT)
print("NumPy version:", np.__version__)
print("Phase 9 synthetic setup: READY")

Project root: C:\Users\Abolfazl.PH\Desktop\cell-irreversibility
NumPy version: 2.0.1
Phase 9 synthetic setup: READY


In [2]:
K = 1.0
DIFFUSION = 1.0
DT = 0.1
N_STEPS = 100_000
SYNTHETIC_SEED = 2031

OMEGA_EQUILIBRIUM = 0.0
OMEGA_NONEQUILIBRIUM = 1.0


stationary_covariance_theory = (
    stationary_covariance_isotropic(
        k=K,
        diffusion=DIFFUSION,
    )
)

sigma_equilibrium_theory = (
    analytic_epr_rotational_ou(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
    )
)

sigma_nonequilibrium_theory = (
    analytic_epr_rotational_ou(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
    )
)

current_equilibrium_theory = (
    analytic_mean_rotational_increment(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
    )
)

current_nonequilibrium_theory = (
    analytic_mean_rotational_increment(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
    )
)


print("Fixed synthetic parameters")
print("--------------------------")
print("k:", K)
print("D:", DIFFUSION)
print("dt:", DT)
print("n_steps:", N_STEPS)
print("seed:", SYNTHETIC_SEED)

print()
print("Theoretical stationary covariance:")
print(stationary_covariance_theory)

print()
print("Theoretical physical EPR")
print("equilibrium:", sigma_equilibrium_theory)
print(
    "nonequilibrium:",
    sigma_nonequilibrium_theory,
)

print()
print("Theoretical mean rotational increment")
print(
    "equilibrium:",
    current_equilibrium_theory,
)
print(
    "nonequilibrium:",
    current_nonequilibrium_theory,
)

Fixed synthetic parameters
--------------------------
k: 1.0
D: 1.0
dt: 0.1
n_steps: 100000
seed: 2031

Theoretical stationary covariance:
[[1. 0.]
 [0. 1.]]

Theoretical physical EPR
equilibrium: 0.0
nonequilibrium: 2.0

Theoretical mean rotational increment
equilibrium: 0.0
nonequilibrium: 0.18066602190484835


In [3]:
equilibrium_path = simulate_rotational_ou(
    n_steps=N_STEPS,
    k=K,
    omega=OMEGA_EQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
    seed=SYNTHETIC_SEED,
)

nonequilibrium_path = simulate_rotational_ou(
    n_steps=N_STEPS,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
    seed=SYNTHETIC_SEED,
)


print(
    "Equilibrium path shape:",
    equilibrium_path.shape,
)

print(
    "Nonequilibrium path shape:",
    nonequilibrium_path.shape,
)

print(
    "All equilibrium values finite:",
    np.isfinite(equilibrium_path).all(),
)

print(
    "All nonequilibrium values finite:",
    np.isfinite(nonequilibrium_path).all(),
)

Equilibrium path shape: (100001, 2)
Nonequilibrium path shape: (100001, 2)
All equilibrium values finite: True
All nonequilibrium values finite: True


In [4]:
equilibrium_covariance_empirical = np.cov(
    equilibrium_path.T,
    ddof=1,
)

nonequilibrium_covariance_empirical = np.cov(
    nonequilibrium_path.T,
    ddof=1,
)


equilibrium_rotational = rotational_increments(
    equilibrium_path
)

nonequilibrium_rotational = rotational_increments(
    nonequilibrium_path
)


equilibrium_current_empirical = (
    equilibrium_rotational.mean()
)

nonequilibrium_current_empirical = (
    nonequilibrium_rotational.mean()
)


synthetic_summary = pd.DataFrame(
    {
        "condition": [
            "equilibrium",
            "nonequilibrium",
        ],
        "omega": [
            OMEGA_EQUILIBRIUM,
            OMEGA_NONEQUILIBRIUM,
        ],
        "theory_epr": [
            sigma_equilibrium_theory,
            sigma_nonequilibrium_theory,
        ],
        "theory_mean_rotational_increment": [
            current_equilibrium_theory,
            current_nonequilibrium_theory,
        ],
        "empirical_mean_rotational_increment": [
            equilibrium_current_empirical,
            nonequilibrium_current_empirical,
        ],
        "empirical_var_x": [
            equilibrium_covariance_empirical[0, 0],
            nonequilibrium_covariance_empirical[0, 0],
        ],
        "empirical_var_y": [
            equilibrium_covariance_empirical[1, 1],
            nonequilibrium_covariance_empirical[1, 1],
        ],
        "empirical_cov_xy": [
            equilibrium_covariance_empirical[0, 1],
            nonequilibrium_covariance_empirical[0, 1],
        ],
    }
)

synthetic_summary

,condition,omega,theory_epr,theory_mean_rotational_increment,empirical_mean_rotational_increment,empirical_var_x,empirical_var_y,empirical_cov_xy
0,equilibrium,0.0,0.0,0.000000,-0.00034,0.992851,0.983713,-0.000967
1,nonequilibrium,1.0,2.0,0.180666,0.17810,0.992347,0.984064,-0.001169


### Initial synthetic benchmark result

The exact-transition simulations reproduce the expected stationary spatial statistics in both conditions.

For the equilibrium process, the empirical covariance is close to the theoretical stationary covariance

$$
C = I,
$$

and the empirical mean signed rotational increment is approximately zero:

$$
\langle c_t\rangle_{\mathrm{emp}}
=
-0.00034.
$$

For the nonequilibrium rotational process, the empirical covariance remains close to the same stationary covariance, but the dynamics display a clear positive rotational current.

The analytical prediction is

$$
\langle c_t\rangle_{\mathrm{theory}}
=
0.180666,
$$

while the simulation gives

$$
\langle c_t\rangle_{\mathrm{emp}}
=
0.17810.
$$

Thus, the equilibrium and nonequilibrium systems can have nearly indistinguishable stationary spatial distributions while exhibiting different temporal probability currents.

This demonstrates why stationary density alone is insufficient to diagnose nonequilibrium dynamics.

The rotational-current observable used here is a dynamical diagnostic and is **not** itself an entropy-production estimate.

## Path-probability-ratio validation criterion

The next benchmark directly compares forward and time-reversed path probabilities for the exactly sampled rotational Ornstein–Uhlenbeck process.

Three rates will be kept distinct:

$$
\sigma_{\mathrm{continuous}}
=
\frac{2\omega^2}{k},
$$

the continuous-time physical entropy-production rate;

$$
\dot I_{\mathrm{sampled,theory}}
=
\frac{
4e^{-2k\,dt}\sin^2(\omega dt)
}{
(1-e^{-2k\,dt})dt
},
$$

the exact forward/reverse path-space irreversibility rate of the process observed only every \(dt\);

and

$$
\dot I_{\mathrm{path,empirical}}
=
\frac{
\log P[\Gamma]
-
\log P[\Gamma^R]
}{
N_{\mathrm{steps}}\,dt
},
$$

the empirical rate calculated from the simulated trajectory.

The validation criteria are fixed before inspecting the empirical path-ratio result.

For the nonequilibrium benchmark:

- the theoretical sampled path-space rate must be positive;
- it must be smaller than the continuous-time physical entropy-production rate at the finite sampling interval \(dt=0.1\);
- the empirical path-log-ratio rate must agree with the exact sampled-theory rate to within **5% relative error**.

For the equilibrium benchmark:

- the theoretical continuous and sampled rates are exactly zero;
- because detailed balance holds analytically, the empirical path-log-ratio rate should be zero up to floating-point numerical error;
- an absolute empirical rate not exceeding \(10^{-10}\) simulation-time\(^{-1}\) will be treated as numerically zero.

These criteria are numerical-validation criteria for the synthetic benchmark. They are not significance thresholds for the later MSC01 experimental analysis.

The 5% tolerance will not be changed after inspection of the empirical path-ratio result merely to obtain a passing validation.

In [5]:
from src.thermo import (
    analytic_sampled_path_irreversibility_rate,
    ou_path_log_ratio,
)


total_time = N_STEPS * DT


equilibrium_path_log_ratio = ou_path_log_ratio(
    path=equilibrium_path,
    k=K,
    omega=OMEGA_EQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)

nonequilibrium_path_log_ratio = ou_path_log_ratio(
    path=nonequilibrium_path,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)


equilibrium_empirical_rate = (
    equilibrium_path_log_ratio
    / total_time
)

nonequilibrium_empirical_rate = (
    nonequilibrium_path_log_ratio
    / total_time
)


equilibrium_sampled_theory = (
    analytic_sampled_path_irreversibility_rate(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
        dt=DT,
    )
)

nonequilibrium_sampled_theory = (
    analytic_sampled_path_irreversibility_rate(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
        dt=DT,
    )
)


print("Total simulated time:", total_time)

print()
print("Equilibrium")
print("-----------")
print(
    "continuous physical EPR:",
    sigma_equilibrium_theory,
)
print(
    "sampled path-space theory:",
    equilibrium_sampled_theory,
)
print(
    "empirical path log ratio:",
    equilibrium_path_log_ratio,
)
print(
    "empirical path-space rate:",
    equilibrium_empirical_rate,
)

print()
print("Nonequilibrium")
print("----------------")
print(
    "continuous physical EPR:",
    sigma_nonequilibrium_theory,
)
print(
    "sampled path-space theory:",
    nonequilibrium_sampled_theory,
)
print(
    "empirical path log ratio:",
    nonequilibrium_path_log_ratio,
)
print(
    "empirical path-space rate:",
    nonequilibrium_empirical_rate,
)

Total simulated time: 10000.0

Equilibrium
-----------
continuous physical EPR: 0.0
sampled path-space theory: 0.0
empirical path log ratio: -1.4551915228366852e-11
empirical path-space rate: -1.4551915228366853e-15

Nonequilibrium
----------------
continuous physical EPR: 2.0
sampled path-space theory: 1.8006480429063036
empirical path log ratio: 17750.719849065208
empirical path-space rate: 1.7750719849065208


In [6]:
nonequilibrium_relative_error = (
    abs(
        nonequilibrium_empirical_rate
        - nonequilibrium_sampled_theory
    )
    / nonequilibrium_sampled_theory
)

equilibrium_absolute_rate = abs(
    equilibrium_empirical_rate
)


path_ratio_validation = pd.DataFrame(
    {
        "condition": [
            "equilibrium",
            "nonequilibrium",
        ],
        "continuous_physical_epr": [
            sigma_equilibrium_theory,
            sigma_nonequilibrium_theory,
        ],
        "sampled_path_theory_rate": [
            equilibrium_sampled_theory,
            nonequilibrium_sampled_theory,
        ],
        "empirical_path_rate": [
            equilibrium_empirical_rate,
            nonequilibrium_empirical_rate,
        ],
    }
)


equilibrium_pass = (
    equilibrium_absolute_rate
    <= 1e-10
)

nonequilibrium_ordering_pass = (
    0.0
    < nonequilibrium_sampled_theory
    < sigma_nonequilibrium_theory
)

nonequilibrium_accuracy_pass = (
    nonequilibrium_relative_error
    <= 0.05
)


print(path_ratio_validation.to_string(index=False))

print()
print(
    "Nonequilibrium relative error:",
    nonequilibrium_relative_error,
)

print()
print("Pre-specified validation checks")
print("--------------------------------")
print(
    "Equilibrium numerical-zero criterion:",
    equilibrium_pass,
)
print(
    "Finite-sampling ordering criterion:",
    nonequilibrium_ordering_pass,
)
print(
    "Nonequilibrium <= 5% relative-error criterion:",
    nonequilibrium_accuracy_pass,
)

print()
print(
    "Overall path-ratio validation:",
    (
        equilibrium_pass
        and nonequilibrium_ordering_pass
        and nonequilibrium_accuracy_pass
    ),
)

     condition  continuous_physical_epr  sampled_path_theory_rate  empirical_path_rate
   equilibrium                      0.0                  0.000000        -1.455192e-15
nonequilibrium                      2.0                  1.800648         1.775072e+00

Nonequilibrium relative error: 0.014203807401751993

Pre-specified validation checks
--------------------------------
Equilibrium numerical-zero criterion: True
Finite-sampling ordering criterion: True
Nonequilibrium <= 5% relative-error criterion: True

Overall path-ratio validation: True


### Path-probability-ratio validation result

The pre-specified synthetic path-ratio validation passed.

For the equilibrium process, the empirical path-space irreversibility rate was

$$
-1.46\times10^{-15},
$$

which is numerically zero and comfortably satisfies the pre-specified absolute tolerance of

$$
10^{-10}.
$$

This is consistent with detailed balance: forward and time-reversed paths have equal probability in the equilibrium benchmark.

For the nonequilibrium rotational process, the continuous-time physical entropy-production rate is

$$
\sigma_{\mathrm{continuous}} = 2.000000.
$$

At the finite observation interval

$$
dt = 0.1,
$$

the exact sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{sampled,theory}}
=
1.800648.
$$

The empirical long-trajectory path-log-ratio calculation gave

$$
\dot I_{\mathrm{path,empirical}}
=
1.775072.
$$

The relative error between the empirical rate and the exact sampled-theory prediction was

$$
0.014204
\approx
1.42\%.
$$

This is below the pre-specified 5% validation tolerance.

All pre-specified validation checks passed.

The difference between the continuous-time physical entropy-production rate and the sampled path-space rate is not a simulation error. It reflects loss of observable temporal information caused by finite-time sampling.

Thus, even with exact stochastic dynamics and exact transition likelihoods, temporal coarse-graining can reduce the irreversibility observable from sampled trajectories.

## Spatial coarse-graining and hidden dissipation

The full two-dimensional rotational Ornstein–Uhlenbeck process is nonequilibrium.

For the pre-specified benchmark,

$$
k=1,\qquad
D=1,\qquad
\omega=1,
$$

the continuous-time physical entropy-production rate is

$$
\sigma_{\mathrm{continuous}}=2.
$$

At the finite observation interval

$$
dt=0.1,
$$

the exact full two-dimensional sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{full,sampled}}
=
1.800648.
$$

However, suppose that only one Cartesian coordinate is observed and the other coordinate is hidden.

For either observed coordinate, the stationary autocovariance is

$$
C_x(\tau)
=
\frac{D}{k}
e^{-k|\tau|}
\cos(\omega\tau).
$$

Because this covariance is invariant under time reversal, the stationary one-coordinate Gaussian process has the same probability for a temporal block and its reversed block.

Therefore,

$$
D_{KL}
\left(
P[x_0,\ldots,x_n]
\parallel
P[x_n,\ldots,x_0]
\right)
=
0.
$$

The same statement holds if only the \(y\) coordinate is observed.

Thus, this benchmark provides an explicit example in which

$$
\text{physical entropy production} > 0
$$

while

$$
\text{observable one-coordinate path irreversibility} = 0.
$$

This is an example of **hidden dissipation under coarse-graining**.

Importantly, the reduced one-coordinate process is not assumed to be first-order Markov. Its complete finite-block Gaussian distribution is used for the numerical check below.

In [7]:
from src.thermo import projected_scalar_path_log_ratio


COARSE_GRAIN_CHECK_STATES = 25

scalar_x_segment = (
    nonequilibrium_path[
        :COARSE_GRAIN_CHECK_STATES,
        0,
    ]
)

scalar_y_segment = (
    nonequilibrium_path[
        :COARSE_GRAIN_CHECK_STATES,
        1,
    ]
)


scalar_x_log_ratio = projected_scalar_path_log_ratio(
    values=scalar_x_segment,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)

scalar_y_log_ratio = projected_scalar_path_log_ratio(
    values=scalar_y_segment,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)


segment_duration = (
    (COARSE_GRAIN_CHECK_STATES - 1)
    * DT
)


scalar_x_rate = (
    scalar_x_log_ratio
    / segment_duration
)

scalar_y_rate = (
    scalar_y_log_ratio
    / segment_duration
)


print(
    "States in scalar check:",
    COARSE_GRAIN_CHECK_STATES,
)

print(
    "Segment duration:",
    segment_duration,
)

print()
print(
    "x-only path log ratio:",
    scalar_x_log_ratio,
)

print(
    "x-only path-space rate:",
    scalar_x_rate,
)

print()
print(
    "y-only path log ratio:",
    scalar_y_log_ratio,
)

print(
    "y-only path-space rate:",
    scalar_y_rate,
)

States in scalar check: 25
Segment duration: 2.4000000000000004

x-only path log ratio: -1.4210854715202004e-14
x-only path-space rate: -5.921189464667501e-15

y-only path log ratio: 1.7763568394002505e-15
y-only path-space rate: 7.401486830834376e-16


In [8]:
coarse_graining_summary = pd.DataFrame(
    {
        "observation": [
            "continuous full system",
            "sampled full 2D",
            "x coordinate only",
            "y coordinate only",
        ],
        "rate": [
            sigma_nonequilibrium_theory,
            nonequilibrium_empirical_rate,
            scalar_x_rate,
            scalar_y_rate,
        ],
        "interpretation": [
            "physical entropy-production rate",
            "empirical sampled path-space irreversibility rate",
            "coarse-grained observed path-space rate",
            "coarse-grained observed path-space rate",
        ],
    }
)

coarse_graining_summary

,observation,rate,interpretation
0,continuous full system,2.000000e+00,physical entropy-production rate
1,sampled full 2D,1.775072e+00,empirical sampled path-space irreversibility rate
2,x coordinate only,-5.921189e-15,coarse-grained observed path-space rate
3,y coordinate only,7.401487e-16,coarse-grained observed path-space rate


### Spatial coarse-graining result

The spatial coarse-graining benchmark produced the analytically expected result.

The complete nonequilibrium system has the continuous-time physical entropy-production rate

$$
\sigma_{\mathrm{continuous}} = 2.
$$

For the full two-dimensional process sampled every

$$
dt = 0.1,
$$

the exact theoretical sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{full,sampled,theory}}
=
1.800648,
$$

and the long simulated trajectory gave

$$
\dot I_{\mathrm{full,sampled,empirical}}
=
1.775072.
$$

However, when either Cartesian coordinate is observed alone, the numerical path-log-ratio rates are

$$
\dot I_x
=
-5.92\times10^{-15}
$$

and

$$
\dot I_y
=
7.40\times10^{-16}.
$$

These values are numerically zero and are consistent with the exact analytical result that the stationary one-coordinate Gaussian process is invariant under time reversal.

Therefore, this synthetic system demonstrates that

$$
\text{physical entropy production} > 0
$$

can coexist with

$$
\text{observable irreversibility in a coarse-grained variable} = 0.
$$

The disappearance of the time-direction signal after hiding one coordinate is an example of **hidden dissipation under coarse-graining**.

This result is especially important for the later MSC01 interpretation: a weak or null trajectory-level irreversibility signal cannot be interpreted as evidence for zero microscopic cellular dissipation.

## Pre-specified validation criteria for the learned DV critic

The next analysis validates the fixed quadratic-logistic Donsker–Varadhan critic on synthetic data before it is applied to MSC01.

The synthetic grouped-sampling design is fixed as:

- 30 independent OU trajectories;
- 800 transitions per trajectory;
- 4 states per path sample;
- 3 transitions per path sample;
- non-overlapping path blocks;
- 200 path samples per trajectory;
- 6000 forward paths;
- 6000 exactly reversed paths;
- trajectory identity as the grouping variable;
- 3-fold GroupKFold;
- master synthetic seed = 2031.

For the nonequilibrium benchmark, the exact sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{sampled,theory}}
=
1.800648
$$

nats per simulation-time unit.

Because each learned path contains three transitions and has duration

$$
\Delta t_{\mathrm{path}}
=
3(0.1)
=
0.3,
$$

the exact path KL divergence is

$$
D_{\mathrm{KL,path}}
=
0.3
\times
1.800648
=
0.540194
$$

nats.

The following validation criteria are fixed before inspecting the learned-critic results.

### Equilibrium benchmark

The held-out weighted raw DV estimate should be close to zero.

The synthetic equilibrium validation will pass if

$$
\left|
\widehat D_{\mathrm{DV,raw}}
\right|
\le
0.05
$$

nats.

### Nonequilibrium benchmark

The learned held-out DV estimate must:

1. be positive;
2. recover at least 50% of the exact path KL;
3. not exceed the exact path KL by more than 0.05 nats.

Therefore,

$$
\widehat D_{\mathrm{DV,raw}}
>
0,
$$

$$
\widehat D_{\mathrm{DV,raw}}
\ge
0.5
D_{\mathrm{KL,path}},
$$

and

$$
\widehat D_{\mathrm{DV,raw}}
\le
D_{\mathrm{KL,path}}
+
0.05.
$$

The allowance above the exact KL is only a finite-sample numerical tolerance; the theoretical Donsker–Varadhan quantity is a lower bound.

### One-coordinate coarse-graining

When either the x coordinate or the y coordinate is observed alone, the exact path-space irreversibility is zero for this synthetic model.

The coarse-grained validation will therefore pass if the weighted raw DV estimate satisfies

$$
\left|
\widehat D_{\mathrm{DV,raw}}
\right|
\le
0.05
$$

nats for each scalar observation.

These thresholds are synthetic numerical-validation criteria only. They are not significance thresholds and will not be transferred to the later MSC01 analysis.

The criteria will not be changed after inspecting the learned-critic results merely to obtain a passing validation.

In [9]:
from src.thermo import (
    build_grouped_ou_path_samples,
    evaluate_grouped_dv_critic,
    summarize_grouped_dv,
)


DV_N_GROUPS = 30
DV_STEPS_PER_GROUP = 800
DV_PATH_N_STATES = 4
DV_N_SPLITS = 3
DV_SEED = 2031

DV_PATH_DURATION = (
    (DV_PATH_N_STATES - 1)
    * DT
)


equilibrium_forward, equilibrium_reverse, equilibrium_groups = (
    build_grouped_ou_path_samples(
        n_groups=DV_N_GROUPS,
        n_steps_per_group=DV_STEPS_PER_GROUP,
        path_n_states=DV_PATH_N_STATES,
        k=K,
        omega=OMEGA_EQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
        seed=DV_SEED,
    )
)

nonequilibrium_forward, nonequilibrium_reverse, nonequilibrium_groups = (
    build_grouped_ou_path_samples(
        n_groups=DV_N_GROUPS,
        n_steps_per_group=DV_STEPS_PER_GROUP,
        path_n_states=DV_PATH_N_STATES,
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
        seed=DV_SEED,
    )
)


nonequilibrium_forward_paths = (
    nonequilibrium_forward.reshape(
        -1,
        DV_PATH_N_STATES,
        2,
    )
)

nonequilibrium_reverse_paths = (
    nonequilibrium_reverse.reshape(
        -1,
        DV_PATH_N_STATES,
        2,
    )
)


nonequilibrium_x_forward = (
    nonequilibrium_forward_paths[:, :, 0]
)

nonequilibrium_x_reverse = (
    nonequilibrium_reverse_paths[:, :, 0]
)

nonequilibrium_y_forward = (
    nonequilibrium_forward_paths[:, :, 1]
)

nonequilibrium_y_reverse = (
    nonequilibrium_reverse_paths[:, :, 1]
)


print("DV path duration:", DV_PATH_DURATION)

print()
print(
    "Equilibrium forward shape:",
    equilibrium_forward.shape,
)
print(
    "Equilibrium reverse shape:",
    equilibrium_reverse.shape,
)
print(
    "Equilibrium groups:",
    np.unique(equilibrium_groups).size,
)

print()
print(
    "Nonequilibrium forward shape:",
    nonequilibrium_forward.shape,
)
print(
    "Nonequilibrium reverse shape:",
    nonequilibrium_reverse.shape,
)
print(
    "Nonequilibrium groups:",
    np.unique(nonequilibrium_groups).size,
)

print()
print(
    "x-only feature shape:",
    nonequilibrium_x_forward.shape,
)
print(
    "y-only feature shape:",
    nonequilibrium_y_forward.shape,
)

DV path duration: 0.30000000000000004

Equilibrium forward shape: (6000, 8)
Equilibrium reverse shape: (6000, 8)
Equilibrium groups: 30

Nonequilibrium forward shape: (6000, 8)
Nonequilibrium reverse shape: (6000, 8)
Nonequilibrium groups: 30

x-only feature shape: (6000, 4)
y-only feature shape: (6000, 4)


In [10]:
equilibrium_dv_folds = evaluate_grouped_dv_critic(
    forward_array=equilibrium_forward,
    reverse_array=equilibrium_reverse,
    groups=equilibrium_groups,
    n_splits=DV_N_SPLITS,
)

nonequilibrium_dv_folds = evaluate_grouped_dv_critic(
    forward_array=nonequilibrium_forward,
    reverse_array=nonequilibrium_reverse,
    groups=nonequilibrium_groups,
    n_splits=DV_N_SPLITS,
)

nonequilibrium_x_dv_folds = evaluate_grouped_dv_critic(
    forward_array=nonequilibrium_x_forward,
    reverse_array=nonequilibrium_x_reverse,
    groups=nonequilibrium_groups,
    n_splits=DV_N_SPLITS,
)

nonequilibrium_y_dv_folds = evaluate_grouped_dv_critic(
    forward_array=nonequilibrium_y_forward,
    reverse_array=nonequilibrium_y_reverse,
    groups=nonequilibrium_groups,
    n_splits=DV_N_SPLITS,
)


fold_summary = pd.concat(
    [
        equilibrium_dv_folds.assign(
            condition="equilibrium full 2D"
        ),
        nonequilibrium_dv_folds.assign(
            condition="nonequilibrium full 2D"
        ),
        nonequilibrium_x_dv_folds.assign(
            condition="nonequilibrium x-only"
        ),
        nonequilibrium_y_dv_folds.assign(
            condition="nonequilibrium y-only"
        ),
    ],
    ignore_index=True,
)


fold_summary[
    [
        "condition",
        "fold",
        "n_train_groups",
        "n_test_groups",
        "group_overlap",
        "n_test_forward",
        "n_test_reverse",
        "dv_raw",
        "dv_clipped",
    ]
]

,condition,fold,n_train_groups,n_test_groups,group_overlap,n_test_forward,n_test_reverse,dv_raw,dv_clipped
0,equilibrium full 2D,1,20,10,0,2000,2000,-0.002289,0.000000
1,equilibrium full 2D,2,20,10,0,2000,2000,-0.007029,0.000000
2,equilibrium full 2D,3,20,10,0,2000,2000,-0.007592,0.000000
3,nonequilibrium full 2D,1,20,10,0,2000,2000,0.554644,0.554644
4,nonequilibrium full 2D,2,20,10,0,2000,2000,0.521502,0.521502
5,nonequilibrium full 2D,3,20,10,0,2000,2000,0.625472,0.625472
6,nonequilibrium x-only,1,20,10,0,2000,2000,-0.000696,0.000000
7,nonequilibrium x-only,2,20,10,0,2000,2000,-0.005340,0.000000
8,nonequilibrium x-only,3,20,10,0,2000,2000,-0.000660,0.000000
9,nonequilibrium y-only,1,20,10,0,2000,2000,0.000432,0.000432


In [11]:
equilibrium_dv_summary = summarize_grouped_dv(
    equilibrium_dv_folds
)

nonequilibrium_dv_summary = summarize_grouped_dv(
    nonequilibrium_dv_folds
)

nonequilibrium_x_dv_summary = summarize_grouped_dv(
    nonequilibrium_x_dv_folds
)

nonequilibrium_y_dv_summary = summarize_grouped_dv(
    nonequilibrium_y_dv_folds
)


exact_path_kl = (
    nonequilibrium_sampled_theory
    * DV_PATH_DURATION
)

nonequilibrium_recovery_fraction = (
    nonequilibrium_dv_summary["dv_raw"]
    / exact_path_kl
)


equilibrium_pass = (
    abs(
        equilibrium_dv_summary["dv_raw"]
    )
    <= 0.05
)

nonequilibrium_positive_pass = (
    nonequilibrium_dv_summary["dv_raw"]
    > 0.0
)

nonequilibrium_recovery_pass = (
    nonequilibrium_dv_summary["dv_raw"]
    >= 0.5 * exact_path_kl
)

nonequilibrium_upper_pass = (
    nonequilibrium_dv_summary["dv_raw"]
    <= exact_path_kl + 0.05
)

x_only_pass = (
    abs(
        nonequilibrium_x_dv_summary["dv_raw"]
    )
    <= 0.05
)

y_only_pass = (
    abs(
        nonequilibrium_y_dv_summary["dv_raw"]
    )
    <= 0.05
)


dv_validation_summary = pd.DataFrame(
    {
        "condition": [
            "equilibrium full 2D",
            "nonequilibrium full 2D",
            "nonequilibrium x-only",
            "nonequilibrium y-only",
        ],
        "dv_raw": [
            equilibrium_dv_summary["dv_raw"],
            nonequilibrium_dv_summary["dv_raw"],
            nonequilibrium_x_dv_summary["dv_raw"],
            nonequilibrium_y_dv_summary["dv_raw"],
        ],
        "dv_clipped": [
            equilibrium_dv_summary["dv_clipped"],
            nonequilibrium_dv_summary["dv_clipped"],
            nonequilibrium_x_dv_summary["dv_clipped"],
            nonequilibrium_y_dv_summary["dv_clipped"],
        ],
    }
)


print("Exact nonequilibrium path KL:", exact_path_kl)

print()
print(dv_validation_summary.to_string(index=False))

print()
print(
    "Nonequilibrium recovery fraction:",
    nonequilibrium_recovery_fraction,
)

print()
print("Pre-specified synthetic DV checks")
print("----------------------------------")
print(
    "Equilibrium |DV raw| <= 0.05:",
    equilibrium_pass,
)
print(
    "Nonequilibrium DV raw > 0:",
    nonequilibrium_positive_pass,
)
print(
    "Nonequilibrium recovery >= 50%:",
    nonequilibrium_recovery_pass,
)
print(
    "Nonequilibrium DV <= exact KL + 0.05:",
    nonequilibrium_upper_pass,
)
print(
    "x-only |DV raw| <= 0.05:",
    x_only_pass,
)
print(
    "y-only |DV raw| <= 0.05:",
    y_only_pass,
)

overall_dv_validation = all(
    [
        equilibrium_pass,
        nonequilibrium_positive_pass,
        nonequilibrium_recovery_pass,
        nonequilibrium_upper_pass,
        x_only_pass,
        y_only_pass,
    ]
)

print()
print(
    "Overall learned-DV validation:",
    overall_dv_validation,
)

Exact nonequilibrium path KL: 0.5401944128718912

             condition    dv_raw  dv_clipped
   equilibrium full 2D -0.005637    0.000000
nonequilibrium full 2D  0.567206    0.567206
 nonequilibrium x-only -0.002232    0.000000
 nonequilibrium y-only -0.000251    0.000000

Nonequilibrium recovery fraction: 1.0500033068222152

Pre-specified synthetic DV checks
----------------------------------
Equilibrium |DV raw| <= 0.05: True
Nonequilibrium DV raw > 0: True
Nonequilibrium recovery >= 50%: True
Nonequilibrium DV <= exact KL + 0.05: True
x-only |DV raw| <= 0.05: True
y-only |DV raw| <= 0.05: True

Overall learned-DV validation: True


### Learned variational-KL validation result

The fixed quadratic-logistic Donsker–Varadhan critic passed all pre-specified synthetic validation criteria.

All cross-validation folds preserved trajectory independence:

- 20 training trajectory groups per fold;
- 10 held-out trajectory groups per fold;
- zero train/test group overlap;
- 2000 held-out forward paths and 2000 held-out reversed paths per fold.

For the equilibrium full two-dimensional process, the weighted held-out raw DV estimate was

$$
\widehat D_{\mathrm{DV,eq}}
=
-0.005637
$$

nats.

This is close to zero and satisfies the pre-specified equilibrium criterion

$$
\left|
\widehat D_{\mathrm{DV,eq}}
\right|
\le
0.05.
$$

For the nonequilibrium full two-dimensional process, the exact path KL divergence for a three-transition path is

$$
D_{\mathrm{KL,path}}
=
0.540194
$$

nats.

The learned held-out DV estimate was

$$
\widehat D_{\mathrm{DV,neq}}
=
0.567206
$$

nats.

The empirical estimate is slightly above the exact population KL value by approximately

$$
0.0270
$$

nats. This does not imply that the theoretical DV lower bound exceeds the true KL divergence. The small overshoot is attributed to finite-sample estimation and remains within the pre-specified allowance of 0.05 nats above the exact KL.

For the coarse-grained one-coordinate observations, the weighted raw DV estimates were

$$
\widehat D_{\mathrm{DV},x}
=
-0.002232
$$

and

$$
\widehat D_{\mathrm{DV},y}
=
-0.000251.
$$

Both are numerically close to zero and satisfy the pre-specified coarse-graining criteria.

Therefore, the learned critic reproduces the qualitative and quantitative structure expected from the synthetic benchmark:

- equilibrium dynamics produce no detectable path-space irreversibility;
- the full nonequilibrium two-dimensional dynamics produce a strong positive forward/reverse signal;
- hiding either coordinate removes the observable time-direction signal;
- the learned full-system estimate is consistent with the known exact path KL within the pre-specified finite-sample tolerance.

Overall learned-DV validation: **PASSED**.

This completes the synthetic validation required before applying the variational path-space estimator to MSC01.